# Chronic Kidney Disease Prediction Using Machine Learning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import pickle

In [ ]:
df = pd.read_csv('../dataset/kidney_disease.csv')
df.head()

In [ ]:
print(df.shape)
print(df.columns)
df.info()

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
df.replace('?', np.nan, inplace=True)
df.replace('\t?', np.nan, inplace=True)
df.replace('\tyes', 'yes', inplace=True)
df.replace('\tno', 'no', inplace=True)

In [ ]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip()

In [ ]:
df['pcv'] = pd.to_numeric(df['pcv'], errors='coerce')
df['wc'] = pd.to_numeric(df['wc'], errors='coerce')
df['rc'] = pd.to_numeric(df['rc'], errors='coerce')

In [ ]:
print(df.isnull().sum())

In [ ]:
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns

for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
print(df.isnull().sum())
print(df.isnull().values.any())

In [ ]:
le = LabelEncoder()

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

In [ ]:
print(df.dtypes)

In [ ]:
plt.figure(figsize=(6,5))
sns.countplot(x='classification', data=df)
plt.title('CKD Class Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(18,10))
sns.heatmap(df.corr(), annot=False, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
X = df.drop(['id', 'classification'], axis=1)
y = df['classification']

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
results = {}

In [ ]:
lr = LogisticRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_acc = accuracy_score(y_test, lr_pred)
print('Logistic Regression Accuracy:', lr_acc)
results['Logistic Regression'] = lr_acc

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
dt_acc = accuracy_score(y_test, dt_pred)
print('Decision Tree Accuracy:', dt_acc)
results['Decision Tree'] = dt_acc

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
print('Random Forest Accuracy:', rf_acc)
results['Random Forest'] = rf_acc

In [ ]:
svm = SVC()
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)
svm_acc = accuracy_score(y_test, svm_pred)
print('SVM Accuracy:', svm_acc)
results['SVM'] = svm_acc

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
knn_pred = knn.predict(X_test)
knn_acc = accuracy_score(y_test, knn_pred)
print('KNN Accuracy:', knn_acc)
results['KNN'] = knn_acc

In [ ]:
results_df = pd.DataFrame({
    'Model': results.keys(),
    'Accuracy': results.values()
})

print(results_df)

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(x='Model', y='Accuracy', data=results_df)
plt.title('Model Accuracy Comparison')
plt.show()

In [ ]:
cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Random Forest Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
print(classification_report(y_test, rf_pred))

In [ ]:
cv_scores = cross_val_score(rf, X_scaled, y, cv=5)
print(cv_scores)
print(cv_scores.mean())

In [ ]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 5, 10]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)

In [ ]:
print(grid_search.best_params_)

In [ ]:
best_rf = grid_search.best_estimator_

In [ ]:
best_pred = best_rf.predict(X_test)
final_acc = accuracy_score(y_test, best_pred)
print('Optimized Random Forest Accuracy:', final_acc)

In [ ]:
importance = best_rf.feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance
})

importance_df = importance_df.sort_values(by='Importance', ascending=False)
print(importance_df.head())

In [ ]:
plt.figure(figsize=(10,8))
sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title('Feature Importance')
plt.show()

In [ ]:
pickle.dump(best_rf, open('../models/final_ckd_model.pkl', 'wb'))
pickle.dump(scaler, open('../models/scaler.pkl', 'wb'))

In [ ]:
loaded_model = pickle.load(open('../models/final_ckd_model.pkl', 'rb'))
sample = X_test[0].reshape(1,-1)
prediction = loaded_model.predict(sample)
print('Prediction:', prediction)